# Don’t Stop ’til You Get Enough: Considerations of Audio Duration on Genre Classification in Deep Learning
**Daniel Bruning - 11/07/2026**

## Elevator Pitch

If you walk into a music store or browse for Michael Jackson online, how would you expect his music to be categorized? Under pop? Under R&B? Under local artists if you happened upon a record store in Gary, Indiana? Most artist and song genres appear intuitive if not obvious. Yet, depending on where you are, what you are familiar with, and who is doing the categorizing, our notions of what define a genre and what examples are true examples of a genre shift. Genre may function as a useful, marketable classification, yet the defining boundaries of what specific songs or artists they circumscribe is fluid and contested.

Now consider machine learning systems being tasked with classifying songs. Before a model can learn to classify music, someone must decide what genres exist, what songs belong to each category, and what characteristics distinguish one genre from another. These decisions define the data the model learns from and thus the predictions it makes in concluding where a song fits.

A common assumption in language model training is more data begets better model performance. If a few seconds of audio are useful then even more seconds should be better. Give it the full catalogue to learn from! If a model can identify a genre from a short excerpt then surely providing the full song should provide the best possible accuracy. Yet, the folks behind tuning machine learning systems frequently encounter diminishing returns in such pursuits. Additional data increases computational cost, storage requirements, and training time. Meanwhile, the resulting performance for all the additional resources and effort may become progressively smaller.

This project investigates whether such a point exists in music genre classification. By comparing performance across multiple architectures and clip lengths, this experiment asks a relatively straightforward question:

How much of a song does a model actually need before it can reliably recognize its genre?

If such a threshold exists, additional audio contributes little new information to the classification task beyond it. This experiment was conducted to look at how various deep learning architectures perform when carrying out music genre classification using audio clips of durations between 1 and 30 seconds. It assesses the models' performance, their sensitivity to duration, behaviour specific to each genre, computational efficiency, generalisation properties, and the kinds of errors they make.

## Reproducibility and Source Code

The complete implementation used throughout this project is maintained in the accompanying GitHub repository:

https://github.com/d-bruning/deep-learning-audio-classification

The repository contains the preprocessing pipeline, dataset partitioning procedures, spectrogram generation scripts, model implementations, experiment execution notebooks, analysis notebooks, and supporting configuration files used to generate the results presented here.

To keep this notebook focused on experimental methodology, results, and interpretation, detailed execution instructions are maintained within the repository documentation rather than duplicated here. The repository README provides step-by-step instructions for obtaining the GTZAN and FMA Small datasets, generating duration-specific excerpts, creating Mel spectrograms, executing the experimental runs, and reproducing the analytical outputs reported in this notebook.

This notebook therefore serves as a documented record of the experimental workflow and findings, while the repository serves as the primary reproducibility artifact for regenerating data, rerunning experiments, and reproducing analyses.

## Experimental Design

The project ultimately evolved into a three-stage investigation.

The first stage used the GTZAN dataset to explore whether the amount of available listening context influences music genre classification performance. Audio excerpts ranging from 1 second to 30 seconds were evaluated across multiple deep learning architectures. The goal was to determine whether longer excerpts consistently improved performance and whether those improvements differed across model families.

The second stage repeated the same duration experiment using the substantially larger FMA Small dataset. Rather than simply increasing the amount of training data, this phase served as a replication study. If duration effects observed in GTZAN reflected genuine properties of music classification, similar patterns should appear on a different dataset with different genres, artists, and recordings.

The final stage focused on MobileNetV2. After the comparative experiments identified MobileNetV2 as the strongest-performing architecture on FMA Small, a targeted refinement experiment was conducted to explore how a promising model responds to additional tuning and to demonstrate the full experimental workflow within this notebook.

Across all experiments, models were evaluated using accuracy, precision, recall, F1 score, balanced accuracy, Cohen's kappa, duration sensitivity, parameter efficiency, genre-specific performance, confusion patterns, and generalization behavior.

## The Datasets

The study intentionally uses two datasets. GTZAN provides a widely used benchmark for music genre classification, while FMA Small provides a larger and more diverse collection of recordings. Using both datasets allows findings to be evaluated for consistency rather than relying on a single benchmark.

### Free Music Archive (FMA) Dataset

This project utilizes the Free Music Archive (FMA) dataset developed by
Michaël Defferrard, Kirell Benzi, Pierre Vandergheynst, and Xavier
Bresson. The FMA dataset was created specifically to support research in
music information retrieval and music analysis. The FMA Small subset
contains 8,000 tracks distributed across 8 balanced genre categories.

The dataset is hosted by the Swiss Data Science Center (EPFL and ETHZ).
Metadata are distributed under the Creative Commons Attribution 4.0
International License (CC BY 4.0), while individual audio recordings are
distributed under licenses selected by their respective artists.

Source Dataset:
https://github.com/mdeff/fma

Citation:

Defferrard, M., Benzi, K., Vandergheynst, P., & Bresson, X. (2017).
FMA: A Dataset for Music Analysis. Proceedings of the 18th International
Society for Music Information Retrieval Conference (ISMIR).

https://arxiv.org/abs/1612.01840

### GTZAN
This project uses the GTZAN Music Genre Dataset, originally developed by
George Tzanetakis and Perry Cook (2002). The dataset contains 1,000
30-second audio excerpts spanning 10 musical genres and remains one of
the most widely used benchmarks for music genre recognition research.

Source Dataset:
https://www.kaggle.com/datasets/andradaolteanu/gtzan-dataset-music-genre-classification

Citation:
Tzanetakis, G., & Cook, P. (2002). Musical genre classification of audio
signals. IEEE Transactions on Speech and Audio Processing, 10(5), 293–302.

## Dataset Limitations

Both datasets represent curated genre taxonomies rather than objective musical truth. Genre labels are assigned by people and organizations, meaning category boundaries often overlap. A song classified as Pop might also be described as Rock, Electronic, Hip-Hop, or Folk depending on the cataloging system being used.

GTZAN additionally contains well-documented limitations, including potential recording duplicates, artist overlap, and labeling inconsistencies. FMA Small is larger and more diverse but introduces its own challenges through broader genre definitions and substantial overlap among musical styles. Consequently, results throughout this notebook are interpreted as comparative evaluations within the selected benchmark datasets rather than definitive estimates of real-world genre classification performance.

## Dataset Preparation Decisions

To ensure valid comparison across architectures, durations, and datasets, several design choices were standardized throughout the study.

A fixed song-level train, validation, and test split was created for each dataset and reused across all experimental conditions. This ensured performance differences could not be attributed to changes in dataset composition. All audio excerpts were converted to Mel spectrograms using a common preprocessing workflow. Multiple random seeds were used for every duration condition to reduce sensitivity to stochastic training behavior.

The central independent variable throughout the study was audio duration. By evaluating the same recordings at multiple lengths while holding all other characteristics constant, the experiments isolate how the amount of available listening context influences classification performance.

### Duration-Based Evaluation

The GTZAN and FMA recordings contain substantially more audio than may actually be necessary for genre recognition. If a model can correctly identify a song's genre from only a few seconds of listening, then processing the entire recording may provide little additional benefit while increasing storage requirements, preprocessing time, training cost, and model complexity.

To investigate this question, duration-specific excerpts of 1, 3, 5, 10, 15, 20, 25, and 30 seconds were generated from the same source recordings. Because every excerpt originated from the same song, the experiment could vary the amount of available listening context while holding other recording characteristics constant. This design allowed audio duration to function as the primary independent variable and enabled direct measurement of how additional listening time influences genre-classification performance.

### Center-Based Clip Extraction

To ensure duration comparisons sampled comparable portions of each recording, excerpts were extracted from the center of each song rather than from the beginning or end. For each target duration, the required number of audio samples was calculated and centered within the original recording. If a recording was shorter than the requested duration, the full recording was retained.

Using centered excerpts reduced the likelihood that results would be driven by idiosyncratic properties of song introductions or endings. Many recordings contain fade-ins, fade-outs, silence, spoken introductions, applause, or other transitional segments near their boundaries. Extracting excerpts from the middle of a track increased the probability that each sample contained representative musical content while maintaining a consistent procedure across all durations and datasets.

This design choice was intended to improve comparability between duration conditions by ensuring that 1-second, 3-second, and 30-second excerpts were derived from the same general region of the original recording rather than from different structural sections of the song.

### Spectrogram Representation

Neural networks cannot directly interpret an audio recording in the same way a human listener can. Before a model can learn from music, the audio signal must be transformed into a representation that captures the characteristics of the sound in a machine-readable format.

To accomplish this, all audio excerpts were converted into Mel spectrograms. A Mel spectrogram is a visual representation of sound that shows how frequency content changes over time. Time is represented across the horizontal axis, frequency across the vertical axis, and color intensity indicates the strength of particular frequencies. This transformation converts an audio waveform into an image-like representation while preserving much of the temporal and spectral information that distinguishes one musical genre from another.

Using a common spectrogram representation across all experiments ensured that differences in performance could be attributed primarily to model architecture and audio duration rather than differences in feature extraction methodology. Although the GTZAN dataset distribution included pre-generated spectrogram images, those images were not used in this study. Instead, all GTZAN and FMA Small spectrograms were regenerated using the same Mel-spectrogram pipeline and parameter settings. This decision ensured that every experimental condition, regardless of dataset, duration, or architecture, was derived from a consistent preprocessing workflow and that observed performance differences could not be attributed to differences in spectrogram generation procedures.

### Fixed Data Partitioning

A song-level partitioning strategy was employed to create training, validation, and test sets prior to model development. The resulting split file was reused for all architectures, durations, and experimental runs. This ensured every model was evaluated on identical observations and prevented variation in data partitioning to allow stronger comparisons across architectures or duration conditions.

### Multiple Random Seeds

Each experimental condition was repeated across multiple random seeds to reduce sensitivity to stochastic effects arising from parameter initialization, data shuffling, and optimization. Repeated trials enabled estimation of average performance and variability across runs, providing a more stable characterization of model behavior than any individual training instance. Aggregate statistics derived from these repeated experiments were subsequently used for comparative analysis and interpretation.

### Standardized Evaluation Framework

All architectures were evaluated using the same performance metrics, including accuracy, balanced accuracy, precision, recall, F1 score, top-2 accuracy, top-3 accuracy, and Cohen's kappa. Applying a consistent evaluation framework across models and duration conditions enabled direct comparison of performance trends and supported investigation of whether observed duration effects were specific to a given architecture or broadly consistent across different deep learning approaches.

## Model Selection
The selected architectures were not intended to identify a universally optimal music genre classifier. Instead, they were chosen to represent different approaches to learning from spectrogram representations.

The GTZAN study evaluated MLP, CNN, MobileNetV2, VGG, ShallowResNet, ResidualCNN, CRNN, and CapsNet in order to survey a broad collection of neural network families. These architectures span fully connected, convolutional, residual, recurrent, transfer-learning, and capsule-network designs.

The FMA Small replication study narrowed the architecture set to CNN, MobileNetV2, CRNN, and ViT. These models were selected because they represent distinct modeling philosophies: convolutional learning, transfer learning, temporal sequence modeling, and transformer-based attention. Reducing the architecture set allowed deeper investigation of duration effects while maintaining architectural diversity.

Finally, MobileNetV2 was selected for focused refinement because it emerged as the strongest overall performer in the FMA Small experiments. Rather than introducing an entirely new architecture, the refinement study investigates what can be learned by examining a successful model more closely.

## From Broad Comparisons to Focused Investigation

The experiments described in the previous sections were designed primarily as comparative studies. The goal was not to identify the single best architecture or maximize classification accuracy through extensive hyperparameter tuning. Instead, the objective was to investigate how audio duration influences music genre classification performance across different model families, datasets, and genres under reasonably consistent training conditions.

To support that objective, architectures were configured only to the point where they demonstrated stable learning behavior. Once satisfactory convergence was achieved, hyperparameters were largely held constant throughout the duration experiments. This approach helped ensure that observed differences in performance could be attributed to architecture and clip length rather than extensive model-specific tuning.

The resulting experiments identified MobileNetV2 as an architecture of particular interest. Across the FMA Small study, MobileNetV2 achieved the highest overall classification accuracy, the strongest F1 score, the largest statistically significant duration gain, and the smallest generalization gap. At the same time, the model exhibited no observed overfitting across the evaluated experimental conditions.

These results naturally raise a follow-up question. If MobileNetV2 emerged as the strongest-performing architecture under a standardized experimental configuration, what happens when the focus shifts from broad comparison to targeted refinement?

The following experiment therefore narrows the scope considerably. Rather than evaluating multiple architectures across multiple durations, the notebook examines a single MobileNetV2 configuration using a representative duration condition and random seed. The purpose is not to establish a new benchmark or replace the primary experiments. Instead, the refinement serves as an example of how findings from an exploratory study can motivate a more focused investigation of a promising architecture.

The experiment also provides a complete demonstration of the project workflow, including dataset loading, model construction, training, evaluation, artifact generation, and result interpretation. While the full studies involved hundreds of completed experimental runs, the refinement example illustrates the underlying process without requiring the reader to reproduce the entire experimental campaign.

## Experimental Pipeline Demonstration

The GTZAN and FMA Small studies collectively required hundreds of individual training runs spanning multiple architectures, audio durations, and random seeds. Reproducing the entire experimental campaign within this notebook would be impractical and would obscure the underlying workflow responsible for generating the reported results.

Instead, this section presents a representative experiment using MobileNetV2. The architecture was selected because it emerged as the strongest overall performer in the FMA Small study, achieving the highest overall accuracy (0.508), the largest statistically significant duration gain (+5.8%), and the smallest observed generalization gap (0.074). The purpose of this demonstration is not to establish new experimental findings, but rather to illustrate the complete training and evaluation workflow used throughout the project.

To keep the demonstration computationally manageable while remaining representative of the larger study, a single experimental condition is evaluated:

| Setting | Value |
|----------|----------|
| Architecture | MobileNetV2 |
| Dataset | FMA Small |
| Duration | 15 seconds |
| Input Resolution | 224 × 224 |
| Learning Rate | 1e-5 |
| Random Seed | 42 |
| Evaluation Metrics | Accuracy, Precision, Recall, F1 Score |

This configuration represents a focused follow-on investigation rather than a new comparative study. All other aspects of the experimental framework remain consistent with the methodology used throughout the FMA Small investigation, allowing any observed changes to be attributed primarily to the revised MobileNetV2 training configuration.

## Imports

In [ ]:
# =====================================================
# STANDARD LIBRARY
# =====================================================

from pathlib import Path
from datetime import datetime
import time


# =====================================================
# THIRD-PARTY LIBRARIES
# =====================================================

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
import seaborn as sns

import wandb

from wandb.integration.keras import (
    WandbMetricsLogger
)


# =====================================================
# KERAS CALLBACKS
# =====================================================

from tensorflow.keras.callbacks import (
    EarlyStopping,
    ReduceLROnPlateau,
    ModelCheckpoint
)


# =====================================================
# SKLEARN EVALUATION
# =====================================================

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    top_k_accuracy_score,
    cohen_kappa_score
)

### MobileNetV2 Model

In [ ]:
# =====================================================
# MOBILENETV2 ARCHITECTURE
# =====================================================

from tensorflow.keras.models import Model

from tensorflow.keras.layers import (

    Input,

    GlobalAveragePooling2D,

    Dense,

    Dropout
)

from tensorflow.keras.optimizers import Adam

from tensorflow.keras.applications import (
    MobileNetV2
)

### Experiment Configuration

In [ ]:
# =====================================================
# EXPERIMENT CONFIGURATION
# =====================================================

DATASET_NAME = "FMA Small"

MODEL_NAME = "mobilenetv2"

SEED = 42

DURATION = "15s"

IMG_SIZE = (224, 224)

BATCH_SIZE = 32

EPOCHS = 30

LEARNING_RATE = 1e-5

NUM_CLASSES = 8

CLASS_NAMES = [

    "electronic",
    "experimental",
    "folk",
    "hip-hop",
    "instrumental",
    "international",
    "pop",
    "rock"
]

GENRE_TO_INT = {

    genre: idx

    for idx, genre

    in enumerate(CLASS_NAMES)
}

### Project Paths

In [ ]:
# =====================================================
# PROJECT PATHS
# =====================================================

PROJECT_ROOT = Path.cwd()

DATA_DIR = (
    PROJECT_ROOT /
    "data" /
    "fma_small"
)

METADATA_DIR = (
    DATA_DIR /
    "metadata"
)

SPECTROGRAM_DIR = (
    DATA_DIR /
    "spectrograms"
)

FMA_SPLIT_FILE = (
    METADATA_DIR /
    "fma_song_splits.csv"
)

print(
    f"Dataset: {DATASET_NAME}"
)

print(
    f"Split File Exists: "
    f"{FMA_SPLIT_FILE.exists()}"
)

print(
    f"Spectrogram Directory Exists: "
    f"{SPECTROGRAM_DIR.exists()}"
)